# Phase D â€” GPU Retrain on Colab

End-to-end pipeline: clone repo â†’ ingest HF corpora â†’ derive Stage 1+3 corpora â†’
train Stage 2 (t5-base) + Stage 1 + Stage 3 â†’ export cascade â†’ download.

Acceptance gate: BIRD-100 EX â‰¥ 25%. Per `docs/results/phase-d-gpu-handoff.md`.

## Runtime requirements

- **Colab T4** (free tier) for Stage 1 + Stage 3 (each ~30-60 min).
- **Colab A100 / Kaggle T4 Ã—2** for full Stage 2 t5-base Ã— 5 epochs (~12-24 hr).
- A100 fits t5-base bs=8 grad_accum=16 fp16 comfortably.
- T4 fits t5-small Ã— 5 epochs in ~6-8 hr â€” use as fallback.

## What this notebook ships

1. Clone the public repo.
2. Install ML extras (transformers, datasets, optimum, etc).
3. Stream raw corpora from HuggingFace:
   - `trl-lab/SQaLe-text-to-SQL-dataset` (~511K rows)
   - `NumbersStation/NSText2SQL` (~289K rows)
   - `gretelai/synthetic_text_to_sql` (~100K rows)
   - `xxxbrem/OmniSQL-BIRD` (9K rows)
   - Spider+BIRD dev manifests (uploaded as data fixtures)
4. Run `build-teacher-cache` per source â†’ ultimate v3 corpus (~500K rows).
5. Run `derive-linker-pairs` + `derive-slot-pairs` for Stage 1 + Stage 3.
6. Train all three stages.
7. Export cascade-v3-gpu + run BIRD smoke.
8. Tarball cascade artefacts â†’ upload to Drive / GitHub Releases.

## 1. Clone repo + install

In [ ]:
!git clone https://github.com/xwiz/db-claw.git
%cd db-claw
!git log --oneline | head -5

In [ ]:
# Install Python deps. Colab pre-installs torch + cuda; we add the
# rest via the package's own pyproject.
!pip install -q -e python/semsql_train python/semsql_eval python/semsql_rewriter
!pip install -q transformers==4.45.2 datasets accelerate optimum[onnxruntime] sqlglot click pyarrow huggingface_hub sentencepiece onnxruntime
!python -c 'import torch; print(f"torch={torch.__version__} cuda={torch.cuda.is_available()} dev={torch.cuda.get_device_name(0) if torch.cuda.is_available() else None}")'

## 2. Fetch raw corpora

In [ ]:
from huggingface_hub import hf_hub_download, snapshot_download
import shutil, os, glob
os.makedirs('data', exist_ok=True)

# SQaLe â€” full snapshot (33 parquet shards, ~7GB total)
snapshot_download(
    repo_id='trl-lab/SQaLe-text-to-SQL-dataset',
    repo_type='dataset',
    local_dir='data/sqale',
    allow_patterns=['data/*.parquet'],
)

# NSText2SQL
p = hf_hub_download(
    repo_id='NumbersStation/NSText2SQL',
    filename='train.jsonl',
    repo_type='dataset',
)
shutil.copy(p, 'data/nstext2sql_train.jsonl')

# Gretel synthetic
p = hf_hub_download(
    repo_id='gretelai/synthetic_text_to_sql',
    filename='synthetic_text_to_sql_train.snappy.parquet',
    repo_type='dataset',
)
shutil.copy(p, 'data/gretel_synth_text_to_sql_train.parquet')

# OmniSQL-BIRD
p = hf_hub_download(
    repo_id='xxxbrem/OmniSQL-BIRD',
    filename='train_bird.json',
    repo_type='dataset',
)
shutil.copy(p, 'data/omnisql_bird_train.json')

!ls -la data/

In [ ]:
# Spider + BIRD dev manifests for eval. Upload via Colab UI (Files
# panel) or fetch from your own bucket. The repo doesn't ship them
# because they're licensed datasets.
from google.colab import files
print('Upload spider/dev.json + bird/dev.json as needed for the BIRD eval at the end.')
# uploaded = files.upload()  # uncomment when ready

## 3. Build ultimate v3 corpus (~10-15 min on Colab)

In [ ]:
import glob
sqale_glob = sorted(glob.glob('data/sqale/data/*.parquet'))[0].rsplit('/', 1)[0] + '/*.parquet'
print(f'SQaLe glob: {sqale_glob}')

# SQaLe â€” biggest source
from semsql_train.teacher_cache import (
    build_teacher_cache_from_sqale,
    build_teacher_cache_from_nstext2sql_jsonl,
    build_teacher_cache_from_omnisql,
    build_teacher_cache_from_omnisql_bird_json,
)
from pathlib import Path

s1 = build_teacher_cache_from_sqale(
    out_jsonl=Path('data/skeleton_train_v3_full.jsonl'),
    parquet_glob=sqale_glob,
)
print(f'sqale: {s1.converted}/{s1.total} retention={s1.retention:.1%}')

In [ ]:
s2 = build_teacher_cache_from_nstext2sql_jsonl(
    in_jsonl=Path('data/nstext2sql_train.jsonl'),
    out_jsonl=Path('data/skeleton_train_v3_nstext2sql.jsonl'),
)
print(f'nstext2sql: {s2.converted}/{s2.total} retention={s2.retention:.1%}')

s3 = build_teacher_cache_from_omnisql(
    out_jsonl=Path('data/skeleton_train_v3_gretel.jsonl'),
    parquet_glob='data/gretel_synth_text_to_sql_train.parquet',
    question_key='sql_prompt',
    sql_key='sql',
    db_id_key='domain',
)
print(f'gretel: {s3.converted}/{s3.total} retention={s3.retention:.1%}')

s4 = build_teacher_cache_from_omnisql_bird_json(
    in_json=Path('data/omnisql_bird_train.json'),
    out_jsonl=Path('data/skeleton_train_v3_omnisql_bird.jsonl'),
)
print(f'omnisql-bird: {s4.converted}/{s4.total} retention={s4.retention:.1%}')

In [ ]:
# Concatenate into ultimate corpus.
!cat data/skeleton_train_v3_full.jsonl data/skeleton_train_v3_nstext2sql.jsonl data/skeleton_train_v3_gretel.jsonl data/skeleton_train_v3_omnisql_bird.jsonl > data/skeleton_train_v3_ultimate.jsonl
!wc -l data/skeleton_train_v3_ultimate.jsonl
!grep -c '"INNER JOIN"' data/skeleton_train_v3_ultimate.jsonl
!grep -c '"kind": "fk"' data/skeleton_train_v3_ultimate.jsonl

## 4. Derive Stage 1 + Stage 3 corpora

In [ ]:
# Stage 1 multi-entity linker pairs
!python -m semsql_train derive-linker-pairs \
  --in data/skeleton_train_v3_ultimate.jsonl \
  --out data/linker_train_v3.jsonl \
  --max-rows 200000

# Split eval
import random, json
random.seed(7)
with open('data/linker_train_v3.jsonl') as f: lines = f.readlines()
random.shuffle(lines)
with open('data/linker_eval_v3.jsonl', 'w') as f: f.writelines(lines[:8000])
with open('data/linker_train_v3.jsonl', 'w') as f: f.writelines(lines[8000:])
print(f'linker train={len(lines)-8000} eval=8000')

# Stage 3 slot filler pairs (with rich extractor logic)
!python -m semsql_train derive-slot-pairs \
  --in data/skeleton_train_v3_ultimate.jsonl \
  --out data/slot_train_v3.jsonl \
  --max-rows 200000

with open('data/slot_train_v3.jsonl') as f: lines = f.readlines()
random.shuffle(lines)
with open('data/slot_eval_v3.jsonl', 'w') as f: f.writelines(lines[:8000])
with open('data/slot_train_v3.jsonl', 'w') as f: f.writelines(lines[8000:])
print(f'slot train={len(lines)-8000} eval=8000')

## 5. Train Stage 1 (linker, ~30-60 min T4)

In [ ]:
!python -m semsql_train train \
  --stage linker \
  --base-model distilbert-base-uncased \
  --train data/linker_train_v3.jsonl \
  --eval data/linker_eval_v3.jsonl \
  --epochs 3 \
  --batch-size 64 \
  --out target/linker-v3-gpu

## 6. Train Stage 3 (slot filler, ~30-60 min T4)

In [ ]:
!python -m semsql_train train \
  --stage slot-filler \
  --base-model distilbert-base-uncased \
  --train data/slot_train_v3.jsonl \
  --eval data/slot_eval_v3.jsonl \
  --epochs 3 \
  --batch-size 64 \
  --out target/slot-filler-v3-gpu

## 7. Train Stage 2 (skeleton, biggest job)

Use t5-small Ã— 3 epochs on T4 (~6-8hr) or t5-base Ã— 5 epochs on A100 (~12-24hr).

For T4, the 50K JOIN-balanced subset finishes in ~1-2hr.

In [ ]:
# Build JOIN-balanced 50K subset.
import random
random.seed(0xCAFEF00D)
with open('data/skeleton_train_v3_ultimate.jsonl') as f: lines = f.readlines()
join_lines = [l for l in lines if 'INNER JOIN' in l]
nojoin_lines = [l for l in lines if 'INNER JOIN' not in l]
random.shuffle(join_lines); random.shuffle(nojoin_lines)
sub = join_lines[:25000] + nojoin_lines[:25000]
random.shuffle(sub)
with open('data/skeleton_train_v3_50k_balanced.jsonl', 'w') as f: f.writelines(sub)
print(f'50K subset: {sum(1 for l in sub if "INNER JOIN" in l)} JOINs')

In [ ]:
# T4 path (default). Switch --base-model to t5-base on A100.
!python -m semsql_train train \
  --stage skeleton \
  --base-model t5-small \
  --train data/skeleton_train_v3_50k_balanced.jsonl \
  --eval data/skeleton_eval.jsonl \
  --epochs 3 \
  --batch-size 16 \
  --grad-accum 2 \
  --out target/skeleton-v3-gpu

## 8. Export cascade + tarball

In [ ]:
!python -m semsql_train export-cascade \
  --output-dir target/cascade-v3-gpu \
  --cascade-version v0.3.0-gpu \
  --skeleton-checkpoint target/skeleton-v3-gpu \
  --linker-checkpoint target/linker-v3-gpu \
  --slot-filler-checkpoint target/slot-filler-v3-gpu

# Patch manifest to point at seq2seq dir + quantised stage files.
import json
m = json.load(open('target/cascade-v3-gpu/manifest.json'))
m['skeleton']['path'] = '_skeleton_export'
m['skeleton']['tokenizer'] = '_skeleton_export/tokenizer.json'
m['linker']['path'] = '_linker_export/model_quantized.onnx'
m['linker']['tokenizer'] = '_linker_export/tokenizer.json'
m['slot_filler']['path'] = '_slot_filler_export/model_quantized.onnx'
m['slot_filler']['tokenizer'] = '_slot_filler_export/tokenizer.json'
json.dump(m, open('target/cascade-v3-gpu/manifest.json', 'w'), indent=2)

!tar czf cascade-v3-gpu.tar.gz -C target cascade-v3-gpu
!ls -la cascade-v3-gpu.tar.gz

from google.colab import files
files.download('cascade-v3-gpu.tar.gz')

## 9. (Optional) Local BIRD smoke

Run on your laptop after extracting the tarball. Requires `target/release/semsql.exe` built with `cargo build --release --features onnx -p semsql-cli`.

```bash
tar xzf cascade-v3-gpu.tar.gz -C target/
cp .venv/Lib/site-packages/onnxruntime/capi/onnxruntime.dll target/release/
rm -rf target/spider_graphs/*.semsql
uv run --package semsql-eval python -m semsql_eval spider \
  --questions data/bird/dev.json \
  --db-root data/bird/dev_databases \
  --name bird --limit 100 \
  --semsql-bin target/release/semsql.exe \
  --cascade-manifest target/cascade-v3-gpu/manifest.json \
  --report-json docs/results/v3-gpu-bird100-report.json \
  --graph-cache-dir target/spider_graphs
```

Acceptance gate per `docs/results/phase-d-gpu-handoff.md`:

- t5-small Ã— 3 epochs (T4): expect EX 5-15%
- t5-base Ã— 5 epochs (A100): expect EX 25-35%